# 🔬 Mercor Paper: Baseline vs. Upgraded GRPO on SWE-bench
## Reproducing "Training Frontier Knowledge Work Agents: A 397B RL Training Guide with SkyRL"
> **Paper**: [mercor.com/blog/training-frontier...](https://www.mercor.com/blog/training-frontier-knowledge-work-agents-a-397b-rl-training-guide-with-skyrl/)  
> **Model used here**: `Qwen/Qwen2.5-Coder-0.5B-Instruct` on SWE-bench Verified  
> **Expected runtime**: ~3 minutes on Colab T4 GPU  

### What this notebook does:
1. Runs **Baseline GRPO** (your original `swe_grpo_one_step.py`) and records time + results
2. Runs **Mercor-Upgraded GRPO** (with 3 surgical fixes from the paper) and records time + results  
3. Prints a **side-by-side comparison table** of both approaches

### ⚠️ Before running: Enable GPU
**Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**

In [ ]:
# Install required packages
!pip install -q "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch
# Colab ships torchao which conflicts with peft — remove it
!pip install -q --no-deps torchao 2>/dev/null || true

print("✅ Packages ready")

In [ ]:
import os
import json
import random
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Fetch the utility file from the original project repo
# (MockEnv, generate, first_bash_block, SYSTEM prompt, NO_COMMAND)
if not os.path.exists("utils.py"):
    print("Fetching utils.py from swe_in_prod_vizuara_01...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py"
    )

from utils import NO_COMMAND, SYSTEM, MockEnv, first_bash_block, generate

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU only"
print(f"✅ Device: {device} | Hardware: {gpu_name}")

## Step 1: Load Config & SWE-bench Task
We pick the simplest task from SWE-bench Verified: one file changed, short patch.  
This is the exact same task used in your original `swe_grpo_one_step.py`.